You are evaluating a candidate sentiment model to replace a production baseline. Your goal is to determine whether this model should ship.

“Ship” means: we would choose the candidate model over the baseline for deployment based on the evidence you collect.

Tenem os un modelo nuevo (candidato) que podría reemplazar el modelo actual en producción (baseline). 
Se debe tomar la decisión: cambiar el modelo o mantener el actual?
La trampa: la precisión general (accuracy)puede engañar. Necesitas investigar MÁS PROFUNDAMENTE.

In [ ]:
# Verificar tu versión de Python (debe ser 3.10+)
python --version
# Si necesitas Python 3.10+, descárgalo desde python.org

# Crear un ambiente virtual
python -m venv .venv

# Activar el ambiente
# En Windows:
.venv\Scripts\activate

# Verificar que estás en el ambiente (verás (.venv) en tu terminal)

# Actualizar pip
pip install --upgrade pip

### Step 1 - Install the required dependencies, set up W&B and make sure the python version is 3.10 and above

In [ ]:
# Se tuvo que corregir las dependencias en el ambiente
!pip install wandb datasets transformers evaluate tqdm emoji regex pandas pyarrow scikit-learn nbformat jupyter

Actualizacion del Pytorch para CUDA

NVIDIA GeForce RTX 5070 Ti GPU de Arquitectura Blackwell

La instalación actual: torch 2.11.0+cpu no recponoce la tarjeta gráfica .
Solución: Instalar PyTorch con Soporte para RTX 5070 Ti
 soporte nativo para sm_120.

 rtx-stone, que ha compilado PyTorch específicamente para las GPU RTX 50-series en Windows. Ofrece mayor rendimiento

In [ ]:
# CUDA instalacion
#!pip install rtx-stone[all]
pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Arch list: {torch.cuda.get_arch_list() if torch.cuda.is_available() else 'N/A'}")

PyTorch version: 2.12.0.dev20260408+cu128
CUDA available: True
CUDA version: 12.8
GPU name: NVIDIA GeForce RTX 5070 Ti
Arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [4]:
# Se tuvo que corregir las dependencias en el ambiente
# !pip install wandb datasets transformers evaluate tqdm emoji regex pandas pyarrow scikit-learn nbformat jupyter

   ---------------------------------------- 0.0/25.0 MB ? eta -:--:--
   --- ------------------------------------ 2.4/25.0 MB 11.2 MB/s eta 0:00:03
   ------- -------------------------------- 4.7/25.0 MB 11.9 MB/s eta 0:00:02
   ----------- ---------------------------- 7.3/25.0 MB 11.9 MB/s eta 0:00:02
   --------------- ------------------------ 9.7/25.0 MB 11.8 MB/s eta 0:00:02
   ------------------- -------------------- 12.3/25.0 MB 11.9 MB/s eta 0:00:02
   ----------------------- ---------------- 14.7/25.0 MB 11.8 MB/s eta 0:00:01
   --------------------------- ------------ 17.3/25.0 MB 11.8 MB/s eta 0:00:01
   ------------------------------- -------- 19.9/25.0 MB 11.9 MB/s eta 0:00:01
   ------------------------------------ --- 22.5/25.0 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  24.9/25.0 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 25.0/25.0 MB 11.5 MB/s  0:00:02
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--

In [5]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import emoji
import wandb

from datasets import load_dataset
from transformers import pipeline

SEED = 42
np.random.seed(SEED)

In [6]:
import os, wandb
wandb.login("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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\USUARIO\_netrc
wandb: Currently logged in as: rbasantes (rbasantes-universidad-yachay-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
!python --version

Python 3.10.19


In [8]:
#imports and config:
import re, regex, emoji
import pandas as pd
import numpy as np
import tqdm

import wandb
from datasets import load_dataset
from transformers import pipeline
import evaluate


# WANDB CONFIG
PROJECT = "mlip-lab4-slices-2026"
ENTITY = None
RUN_NAME = "baseline_vs_candidate"

In [9]:
# Models to compare
MODELS = {
    "baseline_model": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "candidate_model":    "LYTinn/finetuning-sentiment-model-tweet-gpt2",
}

In [10]:
# Label normalization for tweet_eval (0/1/2 -> string labels)
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

# Many HF sentiment models output labels like LABEL_0 / LABEL_1 / LABEL_2
HF_LABEL_MAP = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

USE_HF_DATASET = True  # set False to use tweets.csv fallback

### Step 2 - Load a dataset from Hugging Face

In [11]:
if USE_HF_DATASET:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")
    df = pd.DataFrame(ds["test"]).head(500).copy()
    df["label"] = df["label"].map(ID2LABEL)
else:
    df = pd.read_csv("tweets.csv")
    # Ensure it has 'text' and 'label' columns
    df = df.rename(columns={c: c.strip() for c in df.columns})
    assert {"text","label"}.issubset(df.columns), "tweets.csv must include text,label"
    df["label"] = df["label"].astype(str).str.lower()

df = df[["text","label"]].dropna().reset_index(drop=True)
df.head(10)


,text,label
0,@user @user what do these '1/2 naked pics' hav...,neutral
1,OH: “I had a blue penis while I was this” [pla...,neutral
2,"@user @user That's coming, but I think the vic...",neutral
3,I think I may be finally in with the in crowd ...,positive
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative
5,Savchenko now Saakashvili took drug test live ...,neutral
6,How many more days until opening day? 😩,neutral
7,Twitter's #ThankYouObama Shows Heartfelt Grati...,positive
8,All CSG and Fracking all around Australia is t...,neutral
9,@user @user @user @user @user @user take away ...,negative


### Step 3 - Define Failure-Relevant Metadata

#TODO:
In this step, you will create **at least 5** metadata columns that help you slice and analyze model behavior in Weights & Biases (W&B).
These metadata columns should **capture meaningful properties of the data or model behavior that may influence performance**. You can define them using:

1. Value matching (e.g., tweets containing hashtags or mentions)
2. Regex patterns (e.g., negation words, strong sentiment terms like love or hate)
3. Heuristics (e.g., emoji count, all-caps text, tweet length buckets)

Each metadata column should correspond to a potential hypothesis about when or why a model might succeed or fail.
These columns will be propagated through inference and included in the final predictions_table logged to W&B.

After inference, your W&B table (df_long) will contain:
- The original tweet text
- Ground-truth sentiment labels
- Model predictions and confidence scores
- All metadata columns you defined for slicing

You will use these metadata fields in the W&B UI (via the ➕ Filter option) to:
- Create slices of the data
- Compare model behavior across slices
- Identify patterns, weaknesses, or regressions that are not visible in overall accuracy

In [12]:
# Step 3 – Add slicing metadata (text-only)

# TODO: add your own hypothesis-driven metadata here. 
# Here are examples of the kinds of metadata columns you can add & analyse.
# Categories you can explore are: Linguistic, Emotional/semantic, Model-behavioral. Do not reuse the ones given below.
# Ya vienen en el notebook, pero aquí te explico cada uno:

def count_emojis(text: str) -> int:
    return sum(ch in emoji.EMOJI_DATA for ch in str(text))

# SLICE 1: Tweets con muchos emojis
df["emoji_count"] = df["text"].apply(count_emojis).astype(int)
# El modelo es bueno reconociendo sentimiento con emojis? 🤔

# SLICE 2: Tweets con hashtags
df["has_hashtag"] = df["text"].str.contains(r"#\w+", regex=True)
# Confunde el modelo los hashtags con sentimiento? #trending

# SLICE 3: Tweets con menciones (@usuario)
df["has_mention"] = df["text"].str.contains(r"@\w+", regex=True)
# Le afecta mencionar a otras personas?

# SLICE 4: Tweets con negación (not, never, no)
df["has_negation"] = df["text"].str.contains(r"\b(not|never|no)\b", regex=True)
# Entiende el modelo la negación? "I love it" vs "I don't love it"

# SLICE 5: Tweets de longitud inusual
df["length_bucket"] = pd.cut(
    df["text"].str.len(),
    bins=[0, 50, 100, 200, 1000, 10_000],
    labels=["muy_corto", "corto", "medio", "largo", "muy_largo"],
    include_lowest=True
).astype(str)
# El modelo funciona igual con tweets muy largos o muy cortos?

# Example slice definitions (you'll create more later)
def get_slices(df_any: pd.DataFrame):
    return {
        "emoji_gt3": df_any["emoji_count"] > 3,
        "has_negation": df_any["has_negation"] == True,
        "has_hashtag": df_any["has_hashtag"] == True,
    }

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_26804\3068190950.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["has_negation"] = df["text"].str.contains(r"\b(not|never|no)\b", regex=True)


In [14]:
# Transformers requires a backend (PyTorch/TensorFlow/Flax). We'll use PyTorch.
try:
    import torch, transformers, sys
    print("torch:", torch.__version__)
    print("transformers:", transformers.__version__)
    print("CUDA available:", torch.cuda.is_available())

    print("✅ CUDA version:", torch.version.cuda)
    print("✅ GPU device:", torch.cuda.get_device_name(0))
    print("✅ GPU count:", torch.cuda.device_count())

    print("Python:", sys.executable)
except Exception as e:
    raise RuntimeError("Install PyTorch before proceeding: pip install torch torchvision torchaudio") from e

torch: 2.12.0.dev20260408+cu128
transformers: 5.5.1
CUDA available: True
✅ CUDA version: 12.8
✅ GPU device: NVIDIA GeForce RTX 5070 Ti
✅ GPU count: 1
Python: d:\PM_IA\MIA-lab4\.venv\Scripts\python.exe


###  Step 4 – Run Inference (Two Models)

In this step, you'll use two HuggingFace sentiment analysis models to run inference on your dataset:

In [16]:
from tqdm.auto import tqdm

def run_pipeline(model_id: str, texts: list[str]):
    clf = pipeline(
        "text-classification",
        model=model_id,
        truncation=True,
        max_length=128,     # avoid truncation warnings
        framework="pt",
        device=-1           # CPU
    )
    # (Optional) sanity check label mapping for this model
    # print(model_id, clf.model.config.id2label)

    preds, confs = [], []
    for t in tqdm(texts, desc=f"Infer: {model_id}"):
        out = clf(t)[0]
        lbl = HF_LABEL_MAP.get(out["label"], out["label"])
        preds.append(lbl)
        confs.append(float(out["score"]))
    return preds, confs

pred_frames = []
texts = df["text"].tolist()

for model_name, model_id in MODELS.items():
    yhat, conf = run_pipeline(model_id, texts)
    tmp = df.copy()
    tmp["model"] = model_name
    tmp["pred"] = yhat
    tmp["conf"] = conf
    pred_frames.append(tmp)

df_long = pd.concat(pred_frames, ignore_index=True)

# Add a stable example id so reshaping won't silently drop duplicates
df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

df_long.head(20)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Infer: cardiffnlp/twitter-roberta-base-sentiment-latest:   0%|          | 0/500 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: LYTinn/finetuning-sentiment-model-tweet-gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Infer: LYTinn/finetuning-sentiment-model-tweet-gpt2:   0%|          | 0/500 [00:00<?, ?it/s]

,text,label,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,model,pred,conf,ex_id
0,@user @user what do these '1/2 naked pics' hav...,neutral,0,False,True,True,corto,baseline_model,negative,0.804726,113
1,OH: “I had a blue penis while I was this” [pla...,neutral,0,False,False,False,corto,baseline_model,neutral,0.866949,363
2,"@user @user That's coming, but I think the vic...",neutral,0,False,True,False,corto,baseline_model,neutral,0.763724,102
3,I think I may be finally in with the in crowd ...,positive,0,True,True,False,corto,baseline_model,positive,0.774047,305
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,0,False,True,False,medio,baseline_model,neutral,0.416397,160
5,Savchenko now Saakashvili took drug test live ...,neutral,0,False,False,True,medio,baseline_model,neutral,0.496369,389
6,How many more days until opening day? 😩,neutral,1,False,False,False,muy_corto,baseline_model,neutral,0.466571,295
7,Twitter's #ThankYouObama Shows Heartfelt Grati...,positive,0,True,False,False,corto,baseline_model,positive,0.947964,449
8,All CSG and Fracking all around Australia is t...,neutral,0,False,False,False,medio,baseline_model,negative,0.515992,205
9,@user @user @user @user @user @user take away ...,negative,0,False,True,False,medio,baseline_model,neutral,0.646297,73


In [ ]:
# Step 4.5 – Wide-format Table for Model Comparison (Optional but recommended)
# One row per tweet, with baseline + candidate predictions in columns
# TODO: Replace with your metadata
df_wide = df_long.pivot_table(
    index=[
        "ex_id", "text", "label",
        "emoji_count", "has_hashtag", "has_mention", "has_negation", "length_bucket"
    ],
    columns="model",
    values=["pred", "conf"],
    aggfunc="first"
).reset_index()

# Flatten column names (e.g., pred_baseline_model, conf_candidate_model)
df_wide.columns = ["_".join([c for c in col if c]).strip("_") for col in df_wide.columns]

df_wide.head(5)

### Step 5: Compute Metrics (Accuracy + Slice Accuracy + Regression)

In [ ]:
# TODO: Edit to work for your slices

#compute metrics model-wise
from sklearn.metrics import accuracy_score

def compute_accuracy(y_true, y_pred):
    return accuracy_score(list(y_true), list(y_pred))

# Overall accuracy by model (df_long: one row per (tweet, model))
overall = df_long.groupby("model").apply(
    lambda g: compute_accuracy(g["label"], g["pred"]),
    include_groups=False
)

# Slice accuracy table (uses df_long masks)
slice_table = wandb.Table(columns=["slice", "model", "accuracy"])
slice_metrics = {}

for slice_name, mask in get_slices(df_long).items():
    slice_metrics[slice_name] = {}
    for model_name, g in df_long[mask].groupby("model"):
        acc = float(compute_accuracy(g["label"], g["pred"]))
        slice_table.add_data(slice_name, model_name, acc)
        slice_metrics[slice_name][model_name] = acc

In [ ]:
# TODO: Edit to work for your slices


# Regression-aware evaluation (df_eval: one row per tweet, both model outputs) 
# A regression is when the candidate gets something wrong that the baseline got right.
BASELINE = "baseline_model"
CANDIDATE = "candidate_model"

# Ensure ex_id exists (safe even if it already exists)
df_long = df_long.copy()
if "ex_id" not in df_long.columns:
    df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

# Build df_eval with metadata carried through
df_eval = (
    df_long.pivot_table(
        index=[
            "ex_id", "text", "label",
            "emoji_count", "has_hashtag", "has_mention", "has_negation", "length_bucket"
        ],
        columns="model",
        values=["pred", "conf"],
        aggfunc="first"
    )
    .reset_index()
)

# Flatten column names (pred_baseline_model, conf_candidate_model, etc.)
df_eval.columns = ["_".join([c for c in col if c]).strip("_") for col in df_eval.columns]

# Correctness flags
df_eval["baseline_correct"]  = df_eval[f"pred_{BASELINE}"] == df_eval["label"]
df_eval["candidate_correct"] = df_eval[f"pred_{CANDIDATE}"] == df_eval["label"]

# Regression / improvement flags
df_eval["regressed"]   = df_eval["baseline_correct"] & ~df_eval["candidate_correct"]
df_eval["improved"]    = ~df_eval["baseline_correct"] & df_eval["candidate_correct"]
df_eval["both_wrong"]  = ~df_eval["baseline_correct"] & ~df_eval["candidate_correct"]
df_eval["both_correct"]= df_eval["baseline_correct"] & df_eval["candidate_correct"]

# Confidence-conditional regression (candidate is confident AND worse than baseline)
df_eval["confident_regression"] = df_eval["regressed"] & (df_eval[f"conf_{CANDIDATE}"] >= 0.8)

# Global regression metrics
regression_rate = float(df_eval["regressed"].mean())
improvement_rate = float(df_eval["improved"].mean())
conf_reg_rate = float(df_eval["confident_regression"].mean())

print("Regression rate:", regression_rate)
print("Improvement rate:", improvement_rate)
print("Confident regression rate:", conf_reg_rate)

In [ ]:
# TODO: Edit to work with your slices

# Define slices on df_eval (must use columns that exist in df_eval)
def get_slices_eval(df_any):
    return {
        "emoji_gt3": df_any["emoji_count"] > 3,
        "has_negation": df_any["has_negation"] == True,
        "has_hashtag": df_any["has_hashtag"] == True,
        "long_tweets": df_any["length_bucket"].astype(str).isin(["(200, 1000]", "(1000, 10000]"]),
    }

# Slice-level regression metrics table
reg_table = wandb.Table(columns=["slice", "metric", "value"])
reg_metrics = {}

for slice_name, mask in get_slices_eval(df_eval).items():
    g = df_eval[mask]
    if len(g) == 0:
        continue

    reg = float(g["regressed"].mean())
    imp = float(g["improved"].mean())
    conf_reg = float(g["confident_regression"].mean())

    reg_table.add_data(slice_name, "regression_rate", reg)
    reg_table.add_data(slice_name, "improvement_rate", imp)
    reg_table.add_data(slice_name, "confident_regression_rate", conf_reg)

    reg_metrics[slice_name] = {
        "regression_rate": reg,
        "improvement_rate": imp,
        "conf_reg_rate": conf_reg
    }



# Step 6 — #TODO: Log to W&B & Analyse Slices
# (Make sure PROJECT/ENTITY/RUN_NAME exist from Step 1)

In [ ]:
# Step 6: Log to W&B

PROJECT = "mlip-lab4-slices-2026"
ENTITY = None
RUN_NAME = "baseline_vs_candidate"
run = wandb.init(project=PROJECT, entity=ENTITY, name=RUN_NAME)
wandb.log({"predictions_table": wandb.Table(dataframe=df_long)})
wandb.log({"slice_metrics": slice_table})
wandb.log({"regression_metrics": reg_table})
wandb.log({
    "df_eval": wandb.Table(dataframe=df_eval)
})
for model_name, acc in overall.items():
    wandb.summary[f"{model_name}_accuracy"] = float(acc)
wandb.summary["regression_rate"] = regression_rate
wandb.summary["improvement_rate"] = improvement_rate
wandb.summary["confident_regression_rate"] = conf_reg_rate

print("W&B run URL:", run.get_url())
run.finish()

### Instructions: Exploring Slice-Based Evaluation in W&B

# Purpose
In this lab, you are evaluating a candidate sentiment model to decide whether it should replace an existing baseline (production) model.
You have already:
  - run both models on the same dataset
  - logged predictions, confidence scores, and metadata to W&B
  - created metadata that allows you to slice the data
The most important goal is to understand when and why models behave differently.
Overall accuracy alone is often misleading.

# What to do in W&B
1. Open your W&B run
  - Click the project link and open the latest run.
2. Explore the predictions table
  - Go to the Tables tab and open predictions_table.
  - Each row is one tweet × one model.
3. Create and analyze slices (most important)
  - Use filters to create meaningful slices 
    (e.g., negation, emojis, hashtags, long tweets).
  - For each slice:
    - Compare baseline vs candidate performance.
    - Compare slice accuracy to overall accuracy.
    - Inspect a few misclassified examples to identify patterns.
4. Visualize slice performance
  - Open slice_metrics.
  - Create bar charts comparing baseline vs candidate accuracy for at least two slices.
5. Discuss your findings with the TA
  - Explain why slicing reveals issues that overall accuracy hides.
  - Say whether the candidate model should be deployed and why.


In [ ]:
# Students: replace the placeholders below with 1–2 sentence insights
#TODO: Replace this with 1-2 sentence takeaways for each slice.
saved_slice_notes = ["..."]
pd.DataFrame(saved_slice_notes)

### Step 7 - Targeted stress testing with LLMs

TODO: 
In this step, you will use a Large Language Model (LLM) to generate test cases that specifically target a weakness you observed during slicing.

What to do:
1. Choose one slice where you noticed poor performance, regressions, or surprising behavior.
2. Write a short hypothesis (1–2 sentences) explaining why the model might struggle on this slice. Example:
“The model struggles with tweets that use slang and sarcasm.”
3. Use an LLM to generate 10 test cases designed to test this hypothesis.
These can include:
    - subtle or ambiguous cases
    - difficult or adversarial cases
    - small wording changes that affect sentiment
4. Re-run both models on the generated test cases (helper script given below.)
5. Briefly describe what you observed to the TA:
    - Did the same failures appear again?
    - notice any new failure patterns?
    - would this affect your confidence in deploying the model?

Your input can be in the following format:

> Examples:
> - @user @user That’s coming, but I think the victims are going to be Medicaid recipients.
> - I think I may be finally in with the in crowd #mannequinchallenge  #grads2014 @user
> 
> Generate more tweets using slangs.

Use our provided GPTs to start the task: [llm-based-test-case-generator](https://chatgpt.com/g/g-982cylVn2-llm-based-test-case-generator). If you do not have access to GPTs, use the plain ChatGPT or other LLM providers you have access to instead.

In [ ]:
# TODO: Paste your 10 generated tweets here:
generated_slice_description = ""

generated_cases = [""]

In [ ]:
#Helper code to run models on synthetic test cases:

def run_on_generated_tests(texts, models=MODELS):
    rows = []
    for model_name, model_id in models.items():
        clf = pipeline(
            "text-classification",
            model=model_id,
            truncation=True,
            framework="pt",
            device=-1
        )
        for t in texts:
            out = clf(t)[0]
            rows.append({
                "text": t,
                "model": model_name,
                "pred": HF_LABEL_MAP.get(out["label"], out["label"]),
                "conf": float(out["score"])
            })
    return pd.DataFrame(rows)


In [ ]:
generated_df = run_on_generated_tests(generated_cases)
generated_df

In [ ]:
# OPTIONAL: Log synthetic test cases to W&B
wandb.log({
    "synthetic_tests": wandb.Table(dataframe=generated_df)
})